# Basic Real-Time Sign Language Detection w/ Tensorflow

**NOTE:** If running from a different device, make sure to upgrade your fsspec, pandas, and datasets (huggingface API) + decord and tensorflow via these commands:

- `pip install --upgrade fsspec`
- `pip install --upgrade pandas`
- `pip install --upgrade tensorflow`
- `pip install decord`


Then you can check their current versions / upgraded versions with `pip show library_here` (like `pip show tensorflow`).


Note that after upgrading, you have to restart the kernel to use the upgraded versions.


Additionally, ensure that numpy 1.26.4 is being used.

In [1]:
# importing all the necesary libraries
import numpy as np
import cv2 # this is opencv
import os # helps work w/ file paths
import time # we need to take a break between the images so this spaces time apart yesyesyes
import uuid # name image files
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
import tensorflow as tf
# WHY DO I EKEP GETINTG ERRORS HER AISHFOIMHITCS


#%pip install --upgrade numpy pandas scipy matplotlib scikit-learn
from tensorflow.keras import layers, models
from tensorflow.keras.models import Sequential
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
# import pathlib
from PIL import Image

In [3]:
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") # get it from the secrets tab in colab

#### Getting the Dataset

I'm using a dataset from HuggingFace: https://huggingface.co/datasets/ZahidYasinMittha/American-Sign-Language-Dataset


It's 108,618 videos representing 2,208 ASL words, each word with a minimum of 30 videos.

In [4]:
pip install decord

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 100.9 MB/s eta 0:00:00


In [5]:
from datasets import load_dataset
import decord
print(f"Decord version: {decord.__version__}")

Decord version: 0.6.0


In [6]:
print(f"Pandas version: {pd.__version__}")

Pandas version: 2.2.2


In [7]:
pip show decord

Name: decord
Version: 0.6.0
Summary: Decord Video Loader
Home-page: https://github.com/dmlc/decord
Author: 
Author-email: 
License: APACHE
Location: /usr/local/lib/python3.12/dist-packages
Requires: numpy
Required-by: 


In [8]:
from huggingface_hub import hf_hub_download

Link: https://huggingface.co/datasets/ZahidYasinMittha/American-Sign-Language-Dataset/resolve/main/Aslense%20Dataset.csv

In [9]:
csv_path = hf_hub_download(
    repo_id = "ZahidYasinMittha/American-Sign-Language-Dataset",
    filename = "Aslense Dataset.csv",
    repo_type = "dataset"
)

Aslense%20Dataset.csv:   0%|          | 0.00/3.68M [00:00<?, ?B/s]

In [10]:
df = pd.read_csv(csv_path)
df.head(10)

,word,videos
0,a,A.mp4
1,a,a_7.mp4
2,a,a_2.mp4
3,a,a_4.mp4
4,a,a_1.mp4
5,a,a_5.mp4
6,a,a_6.mp4
7,a,a_3.mp4
8,a,A_video_2.mp4
9,a,A_video_0.mp4


In [11]:
print("Total number of videos: ", len(df))
print("Columns: ", df.columns.tolist())

Total number of videos:  108618
Columns:  ['word', 'videos']


In [12]:
df.describe()

,word,videos
count,108618,108618
unique,2207,108617
top,erase,h_5.mp4
freq,160,2


In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 108618 entries, 0 to 108617
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   word    108618 non-null  object
 1   videos  108618 non-null  object
dtypes: object(2)
memory usage: 1.7+ MB


In [14]:
df.isnull().sum() # no null rows phew

,0
word,0
videos,0


Now that we have the dataset, we have to build a streaming DataLoader so that we can access the videos without downloading all 108 thousand videos.

With decord, we can decode each video on the fly.

In [15]:
# sample = next(iter(ds))
# print(sample.keys())

In [16]:
# print(sample.keys())
# print(type(sample["video"]))

In [17]:
import torch
from torch.utils.data import IterableDataset, DataLoader
import io

In [18]:
ds_debug = load_dataset("ZahidYasinMittha/American-Sign-Language-Dataset", streaming = True, split = "train")

for sample in ds_debug:
  print(type(sample["video"]))
  print(dir(sample["video"]))
  print(sample.keys())
  break

README.md:   0%|          | 0.00/3.16k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/22 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/25 [00:00<?, ?it/s]

<class 'torchcodec.decoders._video_decoder.VideoDecoder'>
['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__len__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_begin_stream_seconds', '_cpu_fallback', '_decoder', '_end_stream_seconds', '_get_key_frame_indices', '_getitem_int', '_getitem_slice', '_hf_encoded', '_num_frames', 'cpu_fallback', 'get_all_frames', 'get_frame_at', 'get_frame_played_at', 'get_frames_at', 'get_frames_in_range', 'get_frames_played_at', 'get_frames_played_in_range', 'metadata', 'stream_index']
dict_keys(['video'])


In [40]:
from huggingface_hub import HfFileSystem

import random

In [41]:
fs = HfFileSystem()

In [42]:
all_files = fs.glob("datasets/ZahidYasinMittha/American-Sign-Language-Dataset/part_*/*.mp4")

In [43]:
print(f"Total files found: {len(all_files)}")
print("Sample paths: ", all_files[:5])

Total files found: 108616
Sample paths:  ['datasets/ZahidYasinMittha/American-Sign-Language-Dataset/part_1/000017451997373907346-LIBRARY.mp4', 'datasets/ZahidYasinMittha/American-Sign-Language-Dataset/part_1/0000197996356050556-CELERY.mp4', 'datasets/ZahidYasinMittha/American-Sign-Language-Dataset/part_1/000039681044643247176-PUSH.mp4', 'datasets/ZahidYasinMittha/American-Sign-Language-Dataset/part_1/0001523804663805528-PASSPORT.mp4', 'datasets/ZahidYasinMittha/American-Sign-Language-Dataset/part_1/00015575668519507424-GOVERNMENT.mp4']


In [19]:
class ASLStreamingDataset(IterableDataset):
  def __init__(self, word_to_idx, split = "train", buffer_size = 200, num_frames = 16, frame_size = (112, 112)):
    super().__init__()
    self.word_to_idx = word_to_idx
    self.split = split
    self.buffer_size = buffer_size
    self.num_frames = num_frames
    self.frame_size = frame_size

    self.filename_to_word = {
        row["videos"].strip(): row["word"].strip()
        for _, row in df.iterrows()
    }

  def __iter__(self):
    ds = load_dataset("ZahidYasinMittha/American-Sign-Language-Dataset", streaming = True, split = self.split)

    ds = ds.shuffle(seed = 42, buffer_size = self.buffer_size)

    for sample in ds: # iterate thru the whoel datset
      decoder = sample["video"]

      try:
        filename = os.path.basename(decoder._hf_encoded["path"]) # filename looks up the label & _hf_encoded holds metadata (incl. source filename)
      except Exception:
        continue # if no filename found, skip

      if filename not in self.filename_to_word:
        continue # this means no label found for the file

      word = self.filename_to_word[filename]
      if word not in self.word_to_idx:
        continue

      label_idx = self.word_to_idx[word]

      tensor = self._preprocess(decoder)

      if tensor is None:
        continue # skip corrupted vids

      yield tensor, torch.tensor(label_idx, dtype = torch.long)

  def _preprocess(self, decoder): # this si a videoencoder object, not raw bytes

    try:
      frame_batch = decoder.get_all_frames()
      frames = frame_batch.data

      if frames.shape[0] == 0:
        return None

      T = frames.shape[0] # sample is exactly num_frames
      indices = torch.linspace(0, T - 1, self.num_frames).long()
      frames = frames[indices] # (num_frames, H, W, 3)

      H, W = self.frame_size
      #frames = frames.permute(0, 3, 1, 2).float() # （T， 3， H， W) # why did i suddenly swithc to chinese keyboard accidentally
      frames = frames.float()
      frames = torch.nn.functional.interpolate(
          frames,
          size = (H, W),
          mode = "bilinear",
          align_corners=  False
      )

      # now normalizes again
      frames = frames / 255.0

      # kinetics mean normalization
      mean = torch.tensor([0.43216, 0.394666, 0.37645]).view(1, 3, 1, 1)
      std = torch.tensor([0.22803, 0.22145, 0.216989]).view(1, 3, 1, 1)
      frames = (frames - mean) / std

      # reorder to (3, T, H, W) for 3d cnn
      frames = frames.permute(1, 0, 2, 3).contiguous()
      return frames

    except Exception as e:
      print(f"Failed to preprocess frame {e}")
      return None

In [20]:
for sample in ds_debug:
  dec = sample["video"]
  print(dec._hf_encoded)
  break

{'path': 'hf://datasets/ZahidYasinMittha/American-Sign-Language-Dataset@3a6226f9c8de394a07b6c2e01158f6291897f97b/part_11/train.mp4', 'bytes': None}


In [21]:
unique_words = sorted(df["word"].unique())
word_to_idx = {word: i for i, word in enumerate(unique_words)}
idx_to_word = {i: word for word, i in word_to_idx.items()}

num_classes = len(word_to_idx)
print(f"Number of classes: {num_classes}")

Number of classes: 2207


In [22]:
train_dataset = ASLStreamingDataset(word_to_idx, split = "train", buffer_size = 200)
test_dataset = ASLStreamingDataset(word_to_idx, split = "test", buffer_size = 200)

In [23]:
train_loader = DataLoader(
    train_dataset,
    batch_size = 8, # process 8 vids at a time
    num_workers = 1,
    pin_memory = True # cpu to gpu transfer
)

test_loader = DataLoader(
    test_dataset,
    batch_size = 8, # process 8 vids at a time
    num_workers = 1,
    pin_memory = True # cpu to gpu transfer
)

### Model

2D cnn learns spatial patterns (e.g. edges, shapes, textures)
* 2D conv kernel is a 3x3 patch

3D CNN learns temporal patterns (how things move over time)
* 3D conv kernel is a 3x3x3 patch

R3D_18 = ResNet-18; it's adapted for video.

https://github.com/kryptologyst/Action-Recognition-in-Videos already trained it on 400 classes for action; we'll take his architecture and use it here.

In [24]:
import torch
import torch.nn as nn
from torchvision.models.video import r3d_18, R3D_18_Weights

In [25]:
class ASLClassifier(nn.Module):
  def __init__(self, num_classes, freeze_backbone = True):
    # we use the numbe of classes (num_classes) for the number of asl words (in this case, 2208)
    # the architecture of f3d_18 has a last layer that's trained on 400 kinetic's classes
    # but we'll change that last layer to train on 2208 asl words

    super().__init__()
    self.backbone = r3d_18(weights = R3D_18_Weights.DEFAULT) # r3d 18 weights default gives best available pretrained weights & downloads about 45 megabtes of websites from pytorch models

    if freeze_backbone:
      for param in self.backbone.parameters():
        param.requires_grad = False

    in_features = self.backbone.fc.in_features
    # original fc layer is backbone.fc = Linear(512, 400) but we need to replace the 400 with the number of clsses we have (2208)


    self.backbone.fc = nn.Sequential(
        nn.Dropout(p = 0.3),
        nn.Linear(in_features, num_classes)
    )

  def forward(self, x):
    return self.backbone(x)

  def unfreeze_backbone(self):
    for param in self.backbone.parameters():
      param.requires_grad = True
    print("Backbone unfrozen. All params are now trainable")

In [26]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = ASLClassifier(num_classes=num_classes, freeze_backbone=True)
model = model.to(device)

Using device: cuda
Downloading: "https://download.pytorch.org/models/r3d_18-b3b3357e.pth" to /root/.cache/torch/hub/checkpoints/r3d_18-b3b3357e.pth


100%|██████████| 127M/127M [00:00<00:00, 182MB/s]


In [27]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params: ,}")

Total parameters: 34,298,463
Trainable parameters:  1,132,191


In [28]:
dummy_input = torch.randn(2, 3, 16, 112, 112).to(device)
with torch.no_grad():
  dummy_output = model(dummy_input)
print(f"Output shape: {dummy_output.shape}")

Output shape: torch.Size([2, 2207])


Training loop:
- CrossEntropyLoss loss function
- Adam optimizer
- Gradient scaler (GradScaler)

In [29]:
from torch.cuda.amp import GradScaler, autocast

In [30]:
def train_model(model, train_loader, test_loader, num_classes, num_epochs = 20, lr = 1e-3, unfreeze_epoch = 5, checkpoint_dir = "checkpoints"):
  os.makedirs(checkpoint_dir, exist_ok=True)
  device = next(model.parameters()).device # mkdir on whatever device model's on

  criterion = nn.CrossEntropyLoss(label_smoothing = 0.1)

  optimizer = torch.optim.AdamW(
      filter(lambda p: p.requires_grad, model.parameters()),
      lr = lr,
      weight_decay = 1e-4
  )

  scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
      optimizer,
      T_max = num_epochs,
      eta_min = 1e-6
  )

  scaler = torch.amp.GradScaler('cuda', enabled = torch.cuda.is_available())

  best_val_acc = 0.0

  for epoch in range(1, num_epochs + 1):
    epoch_start = time.time()

    if epoch == unfreeze_epoch:
      model.unfreeze_backbone()
      optimizer = torch.optim.AdamW(
          model.parameters(),
          lr = 1e-5,
          weight_decay = 1e-4
      )
      scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
          optimizer,
          T_max = (num_epochs - unfreeze_epoch),
          eta_min = 1e-7
      )
      print(f"\nEpoch {epoch}: Switched to full fine-tuning\n")

      # - -- - - - - - - -
      model.train()
      # turns on dropout and batch normalization training behavior
      train_loss = 0.0
      train_correct = 0
      train_total = 0

      for step, (videos, labels) in enumerate(train_loader):
        videos = videos.to(device, non_blocking = True)
        labels = labels.to(device, non_blocking = True)

        optimizer.zero_grad()

        with autocast(enabled = torch.cuda.is_available()):
          logits = model(videos)
          loss = criterion(logits, labels)

        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm = 1.0)

        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()
        predicted = logits.argmax(dim = 1)
        train_correct += (predicted == labels).sum().item()
        train_total += labels.size(0)

        if (step + 1) % 50 == 0:
          running_acc = train_correct / train_total * 100
          print(
              f"Epoch {epoch}, step {step + 1} \n"
              f"Loss: {loss.item():.4f} \n"
              f"Accuracy: {running_acc:.2f}%"
          )

      avg_train_loss = train_loss / (step + 1)
      avg_train_acc = train_correct / train_total * 100

      val_loss, val_top1, val_top5 = evaluate(model, test_loader, criterion, device)

      scheduler.step()

      elapsed = time.time() - epoch_start
      current_lr = optimizer.param_groups[0]['lr']

      print(
          f"\n\nEpoch {epoch}/{num_epochs} ({elapsed:.0f}s) \n"
          f"LR: {current_lr:.2e} \n"
          f"Train loss: {avg_train_loss:.4f} | Training accuracy: {avg_train_acc:.2f}% \n"
          f"Validation loss: {val_loss:.4f} | Validation top-1: {val_top1:.2f}%, validation top-5: {val_top5:.2f}%"
      )

      if val_top1 > best_val_acc:
        best_val_acc = val_top1
        checkpoint_path = os.path.join(checkpoint_dir, "best_model.pt")
        torch.save({
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "val_top1": val_top1,
            "word_to_idx": word_to_idx,
        }, checkpoint_path)
        print(f"Saved new best model: validation top-1 {val_top1:.2f}% --> {checkpoint_path}\n")

  print(f"Training complete. Best validation top-1: {best_val_acc:.2f}%")



In [31]:
def evaluate(model, loader, criterion, device):
  model.eval()
  total_loss = 0.0
  top1_correct = 0
  top5_correct = 0
  num_batches = 0
  total = 0

  with torch.no_grad():
    for videos, labels in loader:
      videos = videos.to(device, non_blocking = True)
      labels = labels.to(device, non_blocking = True)

      with autocast(enabled = torch.cuda.is_available()):
        logits = model(videos)
        loss = criterion(logits, labels)

      total_loss += loss.item()
      num_batches += 1

      # top 1 accuracy
      pred_top1 = logits.argmax(dim = 1)
      top1_correct += (pred_top1 == labels).sum().item()

      # top 5 accuracy
      pred_top5 = logits.topk(5, dim = 1).indices
      for i in range(labels.size(0)):
        if labels[i].item() in pred_top5[i].tolist():
          top5_correct += 1

      total += labels.size(0)

    avg_loss = total_loss / max(num_batches, 1)
    top1_acc = top1_correct / max(total, 1) * 100
    top5_acc = top5_correct / max(total, 1) * 100

    return avg_loss, top1_acc, top5_acc


In [32]:
train_model(
    model = model,
    train_loader = train_loader,
    test_loader = test_loader,
    num_classes = num_classes,
    num_epochs = 20,
    lr = 1e-3,
    unfreeze_epoch = 5,
    checkpoint_dir = "checkpoints"
)

Backbone unfrozen. All params are now trainable

Epoch 5: Switched to full fine-tuning



/tmp/ipykernel_806/3429975061.py:53: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled = torch.cuda.is_available()):
/tmp/ipykernel_806/492974364.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled = torch.cuda.is_available()):




Epoch 5/20 (108s) 
LR: 9.89e-06 
Train loss: 8.1062 | Training accuracy: 0.00% 
Validation loss: 8.5642 | Validation top-1: 0.00%, validation top-5: 0.00%
Training complete. Best validation top-1: 0.00%


In [33]:
from collections import defaultdict

In [34]:
def evaluate_per_class(model, loader, idx_to_word, device, top_n = 20):
  # prints percolass accuracy
  # top_n is how many worst-performing classes to print
  model.eval()

  class_correct = defaultdict(int)
  class_total = defaultdict(int)

  with torch.no_grad():
    for videos, labels in loader:
      videos = videos.to(device, non_blocking = True)
      labels = labels.to(device, non_blocking = True)

      with autocast(enabled = torch.cuda.is_available()):
        logits = model(videos)

      predicted = logits.argmax(dim = 1)

      for pred, true_label in zip(predicted, labels):
        true_idx = true_label.item()
        class_total[true_idx] += 1
        class_correct[true_idx] += int(pred.item() == true_idx)

  results = []
  for idx, total in class_total.items():
    acc = class_correct[idx] / total * 100
    word = idx_to_word[idx]
    results.append((word, acc, total))

  results.sort(key = lambda x: x[1])

  print(f"\n{"="*50}")
  print(f"{"Word":<20} {"Accuracy":>10} {"Samples":>10}")
  print(f"{"=" * 50}")

  # - - - - ----- ---------- ------- - - - - - - -
  print(f"\n{top_n} WORST PERFORMING CLASSES -----")
  for word, acc, total in results[:top_n]:
    bar = "-" * int(acc / 5)
    print(f"{word: <20} {acc: >8.1f}%  ({total} samples)  {bar}")

  print(f"\n{top_n} BEST PERFORMING CLASSES -----")
  for word, acc, total in results[-top_n:]:
    bar = "-" * int(acc / 5)
    print(f"{word: <20} {acc: >8.1f}%  ({total} samples)  {bar}")

    # overall stats
    total_correct = sum(class_correct.values())
    total_samples = sum(class_total.values())
    print(f"\nOverall Top-1: {total_correct / total_samples * 100:.2f}%")

  return results

In [35]:
results = evaluate_per_class(model, test_loader, idx_to_word, device, top_n = 20)

/tmp/ipykernel_806/1997547955.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled = torch.cuda.is_available()):



Word                   Accuracy    Samples

20 WORST PERFORMING CLASSES -----
test                      0.0%  (25 samples)  

20 BEST PERFORMING CLASSES -----
test                      0.0%  (25 samples)  

Overall Top-1: 0.00%


checkpointing

In [36]:
def save_checkpoint(model, optimizer, scheduler, epoch, val_top1, word_to_idx, path):
  torch.save({
      "epoch" : epoch,
      "model_state" : model.state_dict(),
      "optimizer" : optimizer.state_dict(),
      "scheduler" : scheduler.state_dict(),
      "val_top1" : val_top1,
      "word_to_idx" : word_to_idx,
  }, path)
  print(f"Checkpoint saved to {path}")


In [37]:
def load_checkpoint(path, model, optimizer = None, scheduler = None):
  checkpoint = torch.load(path, map_location = "cpu")
  # ^ map location = "cpu" loads the weights onto the cpu no matter where they were saved, and then the model is moved to the right device afterwards so yk checkpoint stuff

  model.load_state_dict(checkpoint["model_state"])

  if optimizer is not None:
    optimizer.load_state_dict(checkpoint["optimizer"])

  if scheduler is not None:
    scheduler.load_state_dict(checkpoint["scheduler"])

  epoch = checkpoint["epoch"]
  val_top1 = checkpoint["val_top1"]

  print(f"Resuming now from epoch {epoch}. Reminder: the best validation (top-1) so far: {val_top1:.2f}%")
  return epoch, val_top1

In [38]:
def train_model_resumable(
    model, train_loader, test_loader, num_classes, num_epochs = 20, lr = 1e-3, unfreeze_epoch = 5, checkpoint_dir = "checkpoints", resume = True
):
  os.makedirs(checkpoint_dir, exist_ok = True)
  device = next(model.parameters()).device

  criterion = nn.CrossEntropyLoss(label_smoothing = 0.1)
  optimizer = torch.optim.AdamW(
      filter(lambda p: p.requires_grad, model.parameters()),
      lr = lr, weight_decay = 1e-4
  )

  scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
      optimizer, T_max = num_epochs, eta_min = 1e-6
  )

  scaler = GradScaler(enabled = torch.cuda.is_available())
  start_epoch = 1
  best_val_acc = 0.0

  # -ad fasdhvnaopwihtaopcgh
  best_checkpoint = os.path.join(checkpoint_dir, "best_model.pt")
  last_checkpoint = os.path.join(checkpoint_dir, "last_epoch.pt")

  if resume and os.path.exists(last_checkpoint):
    # resume from the most recent epoch
    start_epoch, best_val_acc = load_checkpoint(
        last_checkpoint, model, optimizer, scheduler
    )

    start_epoch += 1
    print(f"Resuming from epoch {start_epoch}")

  else:
    print("Starting training from scratch again because no previous epoch was loaded")

  for epoch in range(start_epoch, num_epochs + 1):
    if epoch == unfreeze_epoch:
      model.unfreeze_backbone()
      optimizer = torch.optim.AdamW(
          model.parameters(), lr = 1e-5, weight_decay = 1e-4
      )

      scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
          optimizer, T_max = (num_epochs - unfreeze_epoch), eta_min = 1e-7
      )

      save_checkpoint(
          model, optimizer, scheduler, epoch, best_val_acc, word_to_idx, last_checkpoint
      )

      if val_top1 > best_val_acc:
        best_val_acc = val_top1
        save_checkpoint(
            model, optimizer, scheduler, epoch, best_val_acc, word_to_idx, best_checkpoint
        )

In [39]:
model = ASLClassifier(num_classes = num_classes, freeze_backbone = False)

# checkpoint = torch.load("checkpoints/best_model.pt", map_location = "cpu")
# word_to_idx = checkpoint["word_to_idx"]
idx_to_word = {i: w for w, i in word_to_idx.items()}

FileNotFoundError: [Errno 2] No such file or directory: 'checkpoints/best_model.pt'

In [ ]:
# model.load_state_dict(checkpoint["model_state"])
# model = model.to(device)
# model.eval()

In [ ]:
# print(f"Loaded model from epoch {checkpoint["epoch"]}")
# print(f"with validation top-1: {checkpoint["val_top1"]:.2f}%")

---------------